# Self-Attention

**Companion lesson:** https://ml-viz.vercel.app/courses/transformers/01-self-attention

A from-scratch, runnable implementation of the concepts in the lesson.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = '#e2e8f0'
plt.rcParams['axes.labelcolor'] = '#e2e8f0'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#334155'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.color'] = '#1e293b'
plt.rcParams['figure.figsize'] = (8, 5)
np.random.seed(0)

## Scaled dot-product attention, from scratch

$\text{Attention}(Q,K,V)=\text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$. We implement it, run it on a short sequence, and inspect the attention weights.

In [ ]:
def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def attention(Q, K, V, mask=None):
    d_k = Q.shape[-1]
    scores = Q @ K.swapaxes(-1, -2) / np.sqrt(d_k)
    if mask is not None:
        scores = np.where(mask, scores, -1e9)
    weights = softmax(scores, axis=-1)
    return weights @ V, weights

class SelfAttention:
    def __init__(self, d_model, d_k, seed=0):
        rng = np.random.RandomState(seed)
        self.Wq = rng.randn(d_model, d_k)*0.3
        self.Wk = rng.randn(d_model, d_k)*0.3
        self.Wv = rng.randn(d_model, d_k)*0.3
    def __call__(self, X, mask=None):
        return attention(X@self.Wq, X@self.Wk, X@self.Wv, mask)

X = np.random.randn(5, 8)          # 5 tokens, model dim 8
sa = SelfAttention(d_model=8, d_k=8)
out, W = sa(X)
print('output shape:', out.shape, '| attention rows sum to 1:', np.allclose(W.sum(1), 1))

## Worked numeric example (matches the lesson)

Two tokens with identity Q, K and value vectors $[10,0]$, $[0,10]$. Token 1 should keep mostly its own value plus a third of token 2's.

In [ ]:
Q = np.eye(2); K = np.eye(2); V = np.array([[10., 0.], [0., 10.]])
out, W = attention(Q, K, V)
print('attention weights:\n', np.round(W, 3))
print('token 1 output:', np.round(out[0], 2), '  (mostly its own value)')

## Visualizing what attends to what

In [ ]:
plt.imshow(W if False else sa(X)[1], cmap='magma')
plt.colorbar(label='attention weight'); plt.xlabel('key token'); plt.ylabel('query token')
plt.title('Self-attention weights'); plt.show()

## Causal masking for autoregressive models

GPT-style generation forbids attending to the future. A lower-triangular mask sets future scores to $-\infty$ so their softmax weight is 0 — note the upper triangle is blank.

In [ ]:
T = 5
mask = np.tril(np.ones((T, T))).astype(bool)   # True = allowed
_, Wc = sa(X, mask=mask)
plt.imshow(Wc, cmap='magma')
plt.title('Causal attention (upper triangle = 0)'); plt.xlabel('key'); plt.ylabel('query')
plt.colorbar(); plt.show()
print('row 0 attends only to token 0:', np.round(Wc[0], 3))

## Key takeaways

- Attention is `softmax(QKᵀ/√dₖ)·V` — a content-based weighted average of value vectors.
- Every token reaches every other in **one** step, fully in parallel.
- A **causal mask** enables autoregressive generation.
- The √dₖ scaling keeps softmax out of its saturated, low-gradient regime.